# FE5108 Part 3 — Industry Portfolio Analysis

This notebook consolidates Parts **3A–3E** into one reproducible workflow. It uses the tested `part3` package in this repository, so the notebook calculations match the supplied exhibits.

**Data:** 49 Fama–French industry portfolios and the MSCI USA gross total-return index.  
**Portfolio estimation date:** 31 December 2004 unless otherwise stated.  
**Return convention:** monthly total returns in decimal form.

## How to use this notebook

1. Keep this notebook inside the repository's `part3/` folder.
2. From the repository root, run `uv sync`.
3. Run all cells from top to bottom.
4. Adjust only the clearly labelled presentation cells if the group wants different wording or precision.

The original source workbooks are read without modification.

In [ ]:
from pathlib import Path
import html
import sys

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, SVG, display

# Locate the project root even if Jupyter starts in a parent directory.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / 'part3' / '__init__.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the goldman2 repository.')
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)
ROOT

In [ ]:
from part3.data import (
    MSCI_WORKBOOK_NAME,
    WORKBOOK_NAME,
    load_phase1_data,
)
from part3.evaluation import (
    build_phase3b_results,
    build_phase3c_results,
    build_phase3d_results,
)
from part3.optimization import build_phase2_portfolios
from part3.performance import (
    BENCHMARK_LABEL,
    LONG_ONLY_LABEL,
    build_phase3e_from_workbooks,
    interpretation_markdown as phase3e_interpretation,
)
from part3.reporting import DISPLAY_NAMES, weight_table


## 1. Load and validate the data

The loader converts percentage-point observations to decimals, cleans the documented missing-value sentinels, and requires a continuous monthly history from January 1995 through July 2023.

In [ ]:
INDUSTRY_WORKBOOK = ROOT / 'data' / WORKBOOK_NAME
MSCI_WORKBOOK = ROOT / 'data' / MSCI_WORKBOOK_NAME

phase1 = load_phase1_data(INDUSTRY_WORKBOOK)
portfolios = build_phase2_portfolios(phase1.windows['seed_1'])

validation = pd.DataFrame({
    'Value': [
        phase1.report.rows,
        phase1.report.industries,
        phase1.report.start,
        phase1.report.end,
        phase1.report.missing_values,
    ]
}, index=['Monthly observations', 'Industries', 'Start', 'End', 'Missing values'])
display(validation)
display(pd.Series(phase1.report.window_months, name='Months').to_frame())

## Part 3A — Portfolio construction

All portfolios are estimated from January 1995–December 2004. Expected returns are not used.

- **Unconstrained GMV:** $w=\Sigma^{-1}\mathbf{1}/(\mathbf{1}'\Sigma^{-1}\mathbf{1})$; short positions are allowed.
- **Long-only GMV:** minimizes $w'\Sigma w$ subject to $w\geq0$ and $\mathbf{1}'w=1$.
- **Inverse volatility:** weights are proportional to $1/\sigma_i$.

In [ ]:
weights_3a = weight_table(portfolios)
weights_display = weights_3a.map(lambda value: f'{value:.2%}')
display(weights_display)

diagnostics_3a = portfolios.diagnostics.rename(index=DISPLAY_NAMES)[
    ['annualized_volatility', 'gross_exposure', 'nonzero_holdings', 'effective_holdings']
].copy()
diagnostics_display = diagnostics_3a.copy()
diagnostics_display['annualized_volatility'] = diagnostics_display['annualized_volatility'].map(lambda x: f'{x:.2%}')
diagnostics_display['gross_exposure'] = diagnostics_display['gross_exposure'].map(lambda x: f'{x:.2f}x')
diagnostics_display['effective_holdings'] = diagnostics_display['effective_holdings'].map(lambda x: f'{x:.1f}')
display(diagnostics_display)

## Part 3B — In-sample effectiveness

Annualized realized volatility is calculated over the same January 1995–December 2004 estimation period. The unconstrained solution is expected to be the lowest in sample because it is optimized over the broadest feasible set.

In [ ]:
part3b = build_phase3b_results(phase1.windows['seed_1'], portfolios)
part3b_display = part3b.comparison.copy()
part3b_display['annualized_volatility'] = part3b_display['annualized_volatility'].map(lambda x: f'{x:.2%}')
part3b_display['difference_from_lowest_bps'] = part3b_display['difference_from_lowest_bps'].map(lambda x: f'{x:.1f}')
display(part3b_display)

## Part 3C — Out-of-sample effectiveness

The 31 December 2004 weights are held fixed from January 2005 through July 2023. This is the more meaningful test because the covariance structure can change after the estimation sample.

In [ ]:
part3c = build_phase3c_results(phase1.windows['oos_1'], portfolios)
part3c_display = part3c.comparison.copy()
part3c_display['annualized_volatility'] = part3c_display['annualized_volatility'].map(lambda x: f'{x:.2%}')
part3c_display['difference_from_lowest_bps'] = part3c_display['difference_from_lowest_bps'].map(lambda x: f'{x:.1f}')
display(part3c_display)

## Part 3D — Does re-optimization help?

The long-only portfolio is re-estimated using January 2006–December 2015. The original and updated portfolios are then compared over January 2016–July 2023.

In [ ]:
part3d = build_phase3d_results(
    phase1.windows['seed_1'],
    phase1.windows['seed_2'],
    phase1.windows['reopt_test'],
)
part3d_display = part3d.comparison.copy()
for column in ['annualized_volatility', 'relative_change']:
    part3d_display[column] = part3d_display[column].map(lambda x: f'{x:.2%}')
part3d_display['change_vs_original_bps'] = part3d_display['change_vs_original_bps'].map(lambda x: f'{x:.1f}')
display(part3d_display)
display(Markdown(f'**One-way turnover:** {part3d.turnover:.1%}'))

## Part 3E — Cumulative performance and crisis behavior

The original long-only minimum-variance portfolio is compared with the MSCI USA gross total-return index from 31 December 2004 through 31 July 2023. The chart plots $\log_2$ cumulative wealth, which is mathematically equivalent to a base-2 logarithmic vertical axis.

In [ ]:
part3e = build_phase3e_from_workbooks(INDUSTRY_WORKBOOK, MSCI_WORKBOOK)
ending_wealth = part3e.cumulative_wealth.iloc[-1].map(lambda x: f'${x:.2f}').rename('Ending wealth')
display(ending_wealth.to_frame())

crisis_display = part3e.crisis_summary.copy()
for column in ['Strategy return', 'MSCI USA return', 'Relative return']:
    crisis_display[column] = crisis_display[column].map(lambda x: f'{x:.2%}')
crisis_display['Defensive outperformance'] = crisis_display['Defensive outperformance'].map({True: 'Yes', False: 'No'})
display(crisis_display)

In [ ]:
def cumulative_performance_svg(wealth, width=1000, height=440):
    chart = np.log2(wealth)
    left, right, top, bottom = 76, 24, 36, 58
    plot_w, plot_h = width - left - right, height - top - bottom
    values = chart.to_numpy(dtype=float)
    y_min = min(-1.0, np.floor(values.min() * 2) / 2)
    y_max = max(3.0, np.ceil(values.max() * 2) / 2)
    x = np.linspace(left, left + plot_w, len(chart))
    y = lambda value: top + (y_max - value) / (y_max - y_min) * plot_h
    colors = ['#1F4E78', '#A5A5A5']
    pieces = [f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>", "<rect width='100%' height='100%' fill='white'/>"]
    for tick in np.arange(np.ceil(y_min), np.floor(y_max) + 1):
        yy = y(tick)
        pieces.append(f"<line x1='{left}' y1='{yy:.1f}' x2='{left + plot_w}' y2='{yy:.1f}' stroke='#D9E2F3'/>")
        pieces.append(f"<text x='{left - 12}' y='{yy + 4:.1f}' text-anchor='end' font-family='Arial' font-size='12' fill='#555'>{tick:.0f}  (${2 ** tick:g})</text>")
    for year in range(2005, 2024, 2):
        matches = np.flatnonzero(chart.index.year == year)
        if len(matches):
            xx = x[matches[0]]
            pieces.append(f"<line x1='{xx:.1f}' y1='{top}' x2='{xx:.1f}' y2='{top + plot_h}' stroke='#EEF2F7'/>")
            pieces.append(f"<text x='{xx:.1f}' y='{top + plot_h + 24}' text-anchor='middle' font-family='Arial' font-size='11' fill='#555'>{year}</text>")
    for column, color in zip(chart.columns, colors):
        points = ' '.join(f'{xx:.1f},{y(value):.1f}' for xx, value in zip(x, chart[column]))
        pieces.append(f"<polyline points='{points}' fill='none' stroke='{color}' stroke-width='2.5'/>")
    pieces.append(f"<text x='{width / 2}' y='20' text-anchor='middle' font-family='Arial' font-size='16' font-weight='bold' fill='#17365D'>Cumulative performance: growth of $1</text>")
    for position, (column, color) in enumerate(zip(chart.columns, colors)):
        xx = left + 220 + position * 300
        pieces.append(f"<line x1='{xx}' y1='{height - 16}' x2='{xx + 26}' y2='{height - 16}' stroke='{color}' stroke-width='3'/>")
        pieces.append(f"<text x='{xx + 34}' y='{height - 12}' font-family='Arial' font-size='12' fill='#333'>{html.escape(column)}</text>")
    pieces.append(f"<text transform='translate(18 {height / 2}) rotate(-90)' text-anchor='middle' font-family='Arial' font-size='12' fill='#555'>log₂ growth of $1</text>")
    pieces.append('</svg>')
    return SVG(''.join(pieces))

display(cumulative_performance_svg(part3e.cumulative_wealth))

In [ ]:
def formatted_crisis_table(frame):
    table = frame.copy()
    table['Cumulative return'] = table['Cumulative return'].map(lambda x: f'{x:.2%}')
    return table

gfc_html = formatted_crisis_table(part3e.gfc_top10).to_html()
covid_html = formatted_crisis_table(part3e.covid_top10).to_html()
display(HTML(f"<div style='display:flex;gap:32px;align-items:flex-start'><div><h3>GFC: Sep 2008–Feb 2009</h3>{gfc_html}</div><div><h3>COVID: Feb–Mar 2020</h3>{covid_html}</div></div>"))

## Verified results snapshot

Using the supplied data and the implementation in this folder:

- Out-of-sample annualized volatility (Jan 2005–Jul 2023): **14.29% long-only**, **16.75% unconstrained**, and **17.23% inverse volatility**.
- Re-optimization reduced realized volatility over Jan 2016–Jul 2023 from **14.52% to 12.81%**, with **73.5% one-way turnover**.
- $1 invested on 31 Dec 2004 grew to **$4.79** in the original long-only portfolio and **$5.53** in MSCI USA by 31 Jul 2023.
- During the GFC window, the long-only strategy returned **−40.33%** versus **−41.76%** for MSCI USA: modest relative protection of **+1.44%**.
- During the COVID window, it returned **−21.25%** versus **−19.80%** for MSCI USA: relative underperformance of **−1.44%**.

These values are recalculated by the cells above; this section is a convenient handoff summary.

In [ ]:
display(Markdown(phase3e_interpretation(part3e)))

## Excel exhibits and fine-tuning

The formatted exhibits are available beside this notebook:

- [Part 3A — Portfolio weights](outputs/part3a/industry_portfolio_weights.xlsx)
- [Part 3B — In-sample volatility](outputs/part3b/in_sample_volatility.xlsx)
- [Part 3C — Out-of-sample volatility](outputs/part3c/out_of_sample_volatility.xlsx)
- [Part 3D — Re-optimization](outputs/part3d/reoptimization_analysis.xlsx)
- [Part 3E — Performance and crises](outputs/part3e/cumulative_performance_and_crises.xlsx)

Before submission, the group only needs to decide the final narrative tone, number of decimal places, and which supporting tables belong in the main report versus an appendix.